# Dataset preprocessing

This notebook is the single source that prepares the sentiment dataset for the experiments. Training code reads only the processed parquet files written here.

In [ ]:
from pathlib import Path
import json

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name.lower() == "notebook" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "Data"
PROCESSED_DIR = DATA_DIR / "processed"
FIGURES_DIR = PROJECT_ROOT / "Output" / "figures"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
DATASET_SOURCE = "rotten_tomatoes"

PROJECT_ROOT

## Load dataset

Default and only dataset for this project is Rotten Tomatoes sentiment classification.

In [ ]:
def clean_frame(df: pd.DataFrame) -> pd.DataFrame:
    df = df[["text", "label"]].copy()
    df["text"] = df["text"].astype(str).str.strip()
    df["label"] = df["label"].astype(int)
    df = df[df["text"].ne("")].dropna(subset=["text", "label"])
    return df.reset_index(drop=True)


splits = {"train": "train.parquet", "validation": "validation.parquet", "test": "test.parquet"}
base_url = "hf://datasets/cornell-movie-review-data/rotten_tomatoes/"

train_df = clean_frame(pd.read_parquet(base_url + splits["train"]))
validation_df = clean_frame(pd.read_parquet(base_url + splits["validation"]))
test_df = clean_frame(pd.read_parquet(base_url + splits["test"]))

train_df.shape, validation_df.shape, test_df.shape

## Inspect class balance

In [ ]:
summary = pd.concat(
    [
        train_df.assign(split="train"),
        validation_df.assign(split="validation"),
        test_df.assign(split="test"),
    ],
    ignore_index=True,
)

class_balance = summary.groupby(["split", "label"]).size().reset_index(name="count")
class_balance

In [ ]:
plt.figure(figsize=(7, 4))
sns.barplot(data=class_balance, x="split", y="count", hue="label")
plt.title("Label distribution by split")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "data_label_distribution.png", dpi=160)
plt.show()

## Save processed files

In [ ]:
train_df.to_parquet(PROCESSED_DIR / "train.parquet", index=False)
validation_df.to_parquet(PROCESSED_DIR / "validation.parquet", index=False)
test_df.to_parquet(PROCESSED_DIR / "test.parquet", index=False)

metadata = {
    "dataset_source": DATASET_SOURCE,
    "seed": SEED,
    "splits": {
        "train": len(train_df),
        "validation": len(validation_df),
        "test": len(test_df),
    },
    "columns": ["text", "label"],
    "label_mapping": {"0": "negative", "1": "positive"},
}

with open(PROCESSED_DIR / "metadata.json", "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)

metadata